In [ ]:
!pip install jedi
!pip install -q llama-index-core
!pip install -q llama-index-llms-groq
!pip install -q llama-index-readers-file
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-embeddings-instructor

 Setting up your LLM

In [ ]:
from llama_index.llms.groq import Groq
import os

# Key aus Secret holen
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# LLM definieren
llm = Groq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ.get("GROQ_API_KEY")
)

Eigene Daten erstellen

In [ ]:
import os

os.makedirs("/content/data", exist_ok=True)

with open("/content/data/benno_profile.txt", "w") as f:
    f.write("""
Name: Benno

Alter: 21

Wohnort: Deutschland

Aktuelle Situation:
Benno macht gerade einen Data Science Kurs.
Davor war er in einer Ausbildung zum Anwendungsentwickler.

Ziele:
- Job bekommen und Geld verdienen
- Gym aktiv besuchen
- Wohnung kaufen

Interessen:
- Fitness und Gym
- Autos (z.B. Dodge Charger Hellcat)
- Motorräder
- Business und Dropshipping
- Programmieren

Persönlichkeit:
Benno arbeitet daran dinge noch aktiver anzugehen, und erfolgreich in seiner IT und porgrammierer Karriere zu sein


Name: Timmy

Alter: 20

Wohnort: Ungarn

Aktuelle Situation:
Ist Banker

Ziele:
- Bank eröffnen

Interessen:
- Finanzen
- BMW, vor allem der neue M5 und M3

Persönlichkeit:
Kann gut mit leuten reden, und ist relativ extrovertirert, gut in sales
""")

Daten laden

In [ ]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("/content/data").load_data()

Text splitten

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

text_splitter = SentenceSplitter(chunk_size=300, chunk_overlap=50)

nodes = text_splitter.get_nodes_from_documents(documents)

Embeddings

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embeddings = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Vector DB

In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    documents,
    transformations=[text_splitter],
    embed_model=embeddings
)

Prompt verbessern

In [ ]:
# from llama_index.core.prompts import PromptTemplate

# template = """
# Du bist ein Chatbot, der Fragen zu Benno und Timmy beantwortet.

# Context:
# {context_str}

# Instructions:
# - Beantworten Sie die Frage ausschließlich im Kontext.
# - Wenn die Antwort nicht im Kontext enthalten ist, sagen Sie „Ich weiß es nicht“.

# Question: {query_str}
# Answer:
# """

# prompt = PromptTemplate(template)

Query Engine

In [ ]:
# query_engine = index.as_query_engine(
#     llm=llm,
#     text_qa_template=prompt,
#     similarity_top_k=3
# )

# Testen query engine

In [ ]:
# response = query_engine.query("Wie alt ist Benno?")
# print(response)

In [ ]:
# print(query_engine.query("Was sind Bennos Ziele?"))
# print(query_engine.query("Welche Interessen hat Benno?"))
# print(query_engine.query("Mag Benno Motorräder?"))
# print(query_engine.query("Hat Benno ein Haustier?"))

In [ ]:
# while True:
#     frage = input("Du: ")

#     if frage.lower() in ["exit", "quit"]:
#         break

#     antwort = query_engine.query(frage)
#     print("Bot:", antwort)

Retriever

In [ ]:
retriever = index.as_retriever(similarity_top_k=3)

Memory

In [ ]:
from llama_index.core.memory import ChatMemoryBuffer

memory = ChatMemoryBuffer.from_defaults()

System Prompt

In [ ]:
from llama_index.core.base.llms.types import ChatMessage, MessageRole

prefix_messages = [
    ChatMessage(
        role=MessageRole.SYSTEM,
        content="Du bist ein Chatbot, der Fragen zu Benno und Timmy beantwortet.",
    ),
    ChatMessage(
        role=MessageRole.SYSTEM,
        content="Nutze den Kontext und den bisherigen Chatverlauf.",
    ),
    ChatMessage(
        role=MessageRole.SYSTEM,
        content="Antworte kurz, natürlich und direkt.",
    ),
]

ContextChatEngine

In [ ]:
from llama_index.core.chat_engine import ContextChatEngine

rag_chatbot = ContextChatEngine(
    llm=llm,
    retriever=retriever,
    memory=memory,
    prefix_messages=prefix_messages,
)

Testen

In [ ]:
response = rag_chatbot.chat("Wie alt ist Benno?")
print(response.response)

Benno ist 21 Jahre alt.


Echter Chat Loop

In [ ]:
# while True:
#     frage = input("Du: ")

#     if frage.lower() in ["exit", "quit"]:
#         break

#     antwort = rag_chatbot.chat(frage)
#     print("Bot:", antwort.response)

gradio

--- Gradio V1: Simple Chat ---

In [1]:
import gradio as gr

def rag_chat(message, history):
    response = rag_chatbot.chat(message)
    return response.response

demo = gr.ChatInterface(
    fn=rag_chat,
    title="Benno RAG Assistant",
    description="Frag alles über Benno & Timmy",
    examples=[
        "Wie alt ist Benno?",
        "Was sind Bennos Ziele?",
        "Wer ist Timmy?"
    ]
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://15febef7a2fbc1f3fd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


--- Gradio V2: Stateless Chat ---

In [ ]:
def rag_chat_stateless(message, history):
    # Reset memory jedes Mal
    rag_chatbot.reset()

    # History in ChatMessages umwandeln
    chat_history = []
    for msg in history:
        role = MessageRole.USER if msg["role"] == "user" else MessageRole.ASSISTANT
        chat_history.append(ChatMessage(role=role, content=msg["content"]))

    response = rag_chatbot.chat(message, chat_history=chat_history)

    return response.response


demo = gr.ChatInterface(
    fn=rag_chat_stateless,
    title="Benno RAG Assistant",
    description="Frag alles über Benno & Timmy",
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cfbc6a31141dfd0031.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


--- Gradio V3: Custom UI mit Top-K Slider ---

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# Benno RAG Bot")

    top_k = gr.Slider(1, 5, value=3, step=1, label="Top K")

    chatbot = gr.Chatbot()
    msg = gr.Textbox()

    def custom_chat(message, history, k):
        rag_chatbot._retriever._similarity_top_k = int(k)

        rag_chatbot.reset()

        chat_history = []
        for m in history:
            role = MessageRole.USER if m["role"] == "user" else MessageRole.ASSISTANT
            chat_history.append(ChatMessage(role=role, content=m["content"]))

        response = rag_chatbot.chat(message, chat_history=chat_history)

        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": response.response})

        return history

    msg.submit(
        fn=custom_chat,
        inputs=[msg, chatbot, top_k],
        outputs=chatbot
    ).then(lambda: "", None, msg)

demo.launch(share=True)

/tmp/ipykernel_42664/3932200344.py:6: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()
/tmp/ipykernel_42664/3932200344.py:6: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://134f11d381b57b5e7d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
